<a href="https://colab.research.google.com/github/krishnanithesh22-stack/In-Flight-System-Mumbai-to-Jakarta/blob/main/In_Flight_System_Mumbai_to_Jakarta.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [23]:
import requests
import math
import numpy as np
from scipy.interpolate import splprep, splev
from datetime import datetime, timezone
import plotly.graph_objects as go
from IPython.display import display

In [45]:
import folium
from folium.plugins import AntPath, Fullscreen
import requests, math
import numpy as np
from scipy.interpolate import splprep, splev
from datetime import datetime, timezone
import plotly.graph_objects as go
from IPython.display import display


AIRLINE   = "Garuda Indonesia"
FLIGHT_NO = "GA65"
CALLSIGN  = "GIA65"


ORIGIN = {"name": "Chhatrapati Shivaji Maharaj Intl (BOM)", "lat": 19.0896, "lon": 72.8656, "city": "Mumbai"}
DEST   = {"name": "Soekarno-Hatta International Airport (CGK)",             "lat": -6.1256, "lon": 106.6559, "city": "Jakarta"}


waypoints = [
    (ORIGIN["lat"], ORIGIN["lon"]),
    (15.5, 80.0),
    (11.0, 88.0),
    (6.5, 95.5),
    (2.5, 101.0),
    (-1.0, 104.5),
    (-4.5, 106.0),
    (DEST["lat"], DEST["lon"]),
]


def smooth_route(points, n=300):
    lats = [p[0] for p in points]
    lons = [p[1] for p in points]
    tck, u = splprep([lons, lats], s=0, k=3)
    unew = np.linspace(0, 1, n)
    lon_s, lat_s = splev(unew, tck)
    return list(zip(lat_s, lon_s))


route = smooth_route(waypoints)


def haversine(a, b):
    R = 6371.0
    lat1, lon1 = math.radians(a[0]), math.radians(a[1])
    lat2, lon2 = math.radians(b[0]), math.radians(b[1])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    h = math.sin(dlat/2)**2 + math.cos(lat1)*math.cos(lat2)*math.sin(dlon/2)**2
    return 2*R*math.asin(math.sqrt(h))


def bearing(a, b):
    lat1, lon1 = math.radians(a[0]), math.radians(a[1])
    lat2, lon2 = math.radians(b[0]), math.radians(b[1])
    dlon = lon2 - lon1
    x = math.sin(dlon)*math.cos(lat2)
    y = math.cos(lat1)*math.sin(lat2) - math.sin(lat1)*math.cos(lat2)*math.cos(dlon)
    return (math.degrees(math.atan2(x, y)) + 360) % 360


cum = [0.0]
for i in range(1, len(route)):
    cum.append(cum[-1] + haversine(route[i-1], route[i]))
total_km = cum[-1]


def point_at_fraction(frac):
    target = frac * total_km
    for i in range(1, len(cum)):
        if cum[i] >= target:
            seg_len = cum[i] - cum[i-1]
            seg_frac = 0 if seg_len == 0 else (target - cum[i-1]) / seg_len
            lat = route[i-1][0] + seg_frac*(route[i][0]-route[i-1][0])
            lon = route[i-1][1] + seg_frac*(route[i][1]-route[i-1][1])
            hdg = bearing(route[i-1], route[i])
            return (lat, lon), hdg
    return route[-1], 0


FLIGHT_DURATION_MIN = 8*60 + 10


def simulate_progress():
    now = datetime.now(timezone.utc)
    epoch_min = now.hour*60 + now.minute
    return (epoch_min % FLIGHT_DURATION_MIN) / FLIGHT_DURATION_MIN


def flight_profile(t):
    if t <= 0.01:
        return "At Gate / Departing Mumbai (BOM)", 0, 0
    elif t < 0.10:
        p = t/0.10
        return "Climbing out of Mumbai", int(38000*p), int(180+(450-180)*p)
    elif t < 0.88:
        return "Cruising", 37000+int(2000*math.sin(t*20)), 500+int(20*math.sin(t*15))
    elif t < 0.99:
        p = (t-0.88)/0.11
        return "Descending into Jakarta", int(38000*(1-p)), int(500-(500-180)*p)
    else:
        return "Arrived at Jakarta (CGK)", 0, 0


def get_live_opensky(callsign):
    try:
        r = requests.get("https://opensky-network.org/api/states/all", timeout=6)
        r.raise_for_status()
        for s in r.json().get("states", []) or []:
            if s[1] and s[1].strip() == callsign:
                return {"lat": s[6], "lon": s[5], "alt_m": s[7], "velocity_ms": s[9], "heading": s[10]}
    except Exception:
        pass
    return None


live = get_live_opensky(CALLSIGN)


if live and live["lat"] and live["lon"]:
    cur_pos   = (live["lat"], live["lon"])
    hdg       = live["heading"] or 0
    alt_ft    = int((live["alt_m"] or 0) * 3.28084)
    spd_kt    = int((live["velocity_ms"] or 0) * 1.94384)
    status    = "Cruising" if alt_ft > 1000 else "On ground / near airport"
    data_mode = "LIVE (OpenSky ADS-B)"
else:
    t = simulate_progress()
    cur_pos, hdg = point_at_fraction(t)
    status, alt_ft, spd_kt = flight_profile(t)
    data_mode = "SIMULATED (no live ADS-B match right now)"


OWM_API_KEY = "3bb8ae2b8f044f9a9afbb21ab6ae4d97"


def get_weather(lat, lon):
    try:
        r = requests.get(
            "https://api.openweathermap.org/data/2.5/weather",
            params={"lat": lat, "lon": lon, "appid": OWM_API_KEY, "units": "metric"},
            timeout=10
        )
        print("OWM status:", r.status_code)
        if r.status_code == 200:
            d = r.json()
            temp = d["main"]["temp"]
            wind = d["wind"]["speed"]
            desc = d["weather"][0]["description"]
            return f"{temp}°C, wind {wind} m/s, {desc} (OpenWeatherMap)"
    except Exception as e:
        print("OWM exception:", e)


    try:
        r = requests.get(
            "https://api.open-meteo.com/v1/forecast",
            params={"latitude": lat, "longitude": lon, "current_weather": "true"},
            timeout=10
        )
        print("Open-Meteo status:", r.status_code)
        r.raise_for_status()
        cw = r.json()["current_weather"]
        return f"{cw['temperature']}°C, wind {cw['windspeed']} km/h (Open-Meteo)"
    except Exception as e:
        print("Open-Meteo exception:", e)
        return f"unavailable ({e})"


result = get_weather(19.0896, 72.8656)
print("RESULT:", result)


print(f"Data mode: {data_mode}")
print(f"Status: {status} | Alt: {alt_ft} ft | Speed: {spd_kt} kt | Pos: {cur_pos}")


route_lats = [p[0] for p in route]
route_lons = [p[1] for p in route]


fig = go.Figure()


fig.add_trace(go.Scattergeo(
    lon=route_lons, lat=route_lats, mode="lines",
    line=dict(width=2, color="crimson"), name="Route"))


fig.add_trace(go.Scattergeo(
    lon=[ORIGIN["lon"], DEST["lon"]], lat=[ORIGIN["lat"], DEST["lat"]],
    mode="markers+text", text=["BOM", "CGK"], textposition="top center", textfont=dict(color="red", size=13),
    marker=dict(size=8, color="blue"), name="Airports"))


fig.add_trace(go.Scattergeo(
    lon=[cur_pos[1]], lat=[cur_pos[0]], mode="markers",
    marker=dict(size=12, color="orange", symbol="triangle-up"),
    name=f"{AIRLINE} {FLIGHT_NO}"))


fig.update_geos(
    projection_type="orthographic",
    showland=True, landcolor="#3a3a3a",
    showocean=True, oceancolor="#0b3d59",
    showcountries=True, countrycolor="#666",
    projection_rotation=dict(lon=(ORIGIN["lon"]+DEST["lon"])/2, lat=5, roll=0),
    center=dict(lon=(ORIGIN["lon"]+DEST["lon"])/2, lat=5))


fig.update_layout(
    title=f"{AIRLINE} {FLIGHT_NO} — BOM to CGK",
    height=650, margin=dict(l=0, r=0, t=40, b=0))


fig.show()



OWM status: 200
RESULT: 28.53°C, wind 8.42 m/s, broken clouds (OpenWeatherMap)
Data mode: SIMULATED (no live ADS-B match right now)
Status: Cruising | Alt: 38903 ft | Speed: 481 kt | Pos: (np.float64(3.3024914344685197), np.float64(99.96740090906519))
